In [1]:
import pandas as pd
import numpy as np

# Configurar aleatoriedad para simulación contable
np.random.seed(42)
n_registros = 1000

# 1. Crear el DataLake / Reporte Base de SAP
id_transacciones = [f"TX-{100000 + i}" for i in range(n_registros)]
fechas = pd.date_range(start="2026-10-01", periods=30, freq="D")
lista_fechas = [np.random.choice(fechas) for _ in range(n_registros)]

unidades_negocio = ["Savory", "Biscuits", "Foods", "C&NB"]
canales_venta = ["DTS", "Organized Trade", "C-Stores", "Wholesale", "Away From Home"]

data_sap = {
    "ID_Transaccion": id_transacciones,
    "Fecha": lista_fechas,
    "Unidad_Negocio": [np.random.choice(unidades_negocio) for _ in range(n_registros)],
    "Canal": [np.random.choice(canales_venta) for _ in range(n_registros)],
    "Monto_SAP": np.round(np.random.uniform(500, 50000, n_registros), 2)
}

df_sap = pd.DataFrame(data_sap)

# 2. Crear el Reporte Espejo de Synapse (MicroStrategy) con discrepancias
df_synapse = df_sap[["ID_Transaccion", "Fecha", "Monto_SAP"]].copy()
df_synapse.columns = ["ID_Transaccion", "Fecha", "Monto_Synapse"]

# Inyectar anomalías contables (Errores de sistema o de registro)
# Caso A: Modificar montos en 15 transacciones (Diferencias de conciliación)
indices_diferencias = np.random.choice(n_registros, 15, replace=False)
df_synapse.loc[indices_diferencias, "Monto_Synapse"] = df_synapse.loc[indices_diferencias, "Monto_Synapse"] * 1.05

# Caso B: Eliminar 5 registros en Synapse (Transacciones que faltan conciliar)
indices_faltantes = np.random.choice(n_registros, 5, replace=False)
df_synapse = df_synapse.drop(indices_faltantes)

print(f"✅ ¡Datos creados con éxito!")
print(f"Registros en SAP: {len(df_sap)} | Registros en Synapse: {len(df_synapse)}")


✅ ¡Datos creados con éxito!
Registros en SAP: 1000 | Registros en Synapse: 995


In [2]:
# 1. Unir ambos reportes usando el ID de Transacción único (Equivalente a un BUSCARV avanzado)
df_conciliacion = pd.merge(df_sap, df_synapse[['ID_Transaccion', 'Monto_Synapse']], on='ID_Transaccion', how='left')

# 2. Calcular la diferencia matemática al centavo
df_conciliacion['Diferencia'] = np.round(df_conciliacion['Monto_SAP'] - df_conciliacion['Monto_Synapse'].fillna(0), 2)

# 3. Clasificar automáticamente el estatus contable de cada línea
condiciones = [
    (df_conciliacion['Monto_Synapse'].isna()),                                     # No llegó a Synapse
    (df_conciliacion['Diferencia'] != 0) & (df_conciliacion['Monto_Synapse'].notna()), # Montos diferentes
    (df_conciliacion['Diferencia'] == 0)                                          # Todo cuadra perfecto
]

valores_estatus = ['Falta en Synapse', 'Diferencia de Monto', 'Conciliado']
df_conciliacion['Estatus_Conciliacion'] = np.select(condiciones, valores_estatus, default='Revisión Requerida')

# 4. Extraer únicamente las discrepancias (el reporte de errores para finanzas)
df_errores = df_conciliacion[df_conciliacion['Estatus_Conciliacion'] != 'Conciliado']

print(f"📊 ¡Conciliación automatizada completada!")
print(f"Total de discrepancias financieras encontradas: {len(df_errores)}")
print("\nPrimeras 5 discrepancias detectadas por el sistema:")
df_errores[['ID_Transaccion', 'Unidad_Negocio', 'Canal', 'Monto_SAP', 'Monto_Synapse', 'Diferencia', 'Estatus_Conciliacion']].head()


📊 ¡Conciliación automatizada completada!
Total de discrepancias financieras encontradas: 20

Primeras 5 discrepancias detectadas por el sistema:


,ID_Transaccion,Unidad_Negocio,Canal,Monto_SAP,Monto_Synapse,Diferencia,Estatus_Conciliacion
33,TX-100033,Biscuits,Away From Home,13292.37,13956.9885,-664.62,Diferencia de Monto
66,TX-100066,Biscuits,Away From Home,40681.73,42715.8165,-2034.09,Diferencia de Monto
156,TX-100156,Foods,C-Stores,42710.04,NaN,42710.04,Falta en Synapse
224,TX-100224,Savory,Away From Home,20496.90,21521.7450,-1024.85,Diferencia de Monto
280,TX-100280,Biscuits,DTS,44899.42,47144.3910,-2244.97,Diferencia de Monto


In [3]:
# Agrupar las discrepancias financieras por Unidad de Negocio y Canal de Venta
flash_report = df_errores.groupby(['Unidad_Negocio', 'Canal']).agg(
    Numero_Errores=('ID_Transaccion', 'count'),
    Total_Desviacion_SAP=('Diferencia', 'sum')
).reset_index()

# Ordenar de mayor a menor impacto financiero para auditoría externa/interna
flash_report = flash_report.sort_values(by='Total_Desviacion_SAP', key=abs, ascending=False)

print("📈 SALES FLASH REPORT - RESUMEN DE COMPROBACIÓN DE AUDITORÍA:")
flash_report


📈 SALES FLASH REPORT - RESUMEN DE COMPROBACIÓN DE AUDITORÍA:


,Unidad_Negocio,Canal,Numero_Errores,Total_Desviacion_SAP
7,Foods,C-Stores,1,42710.04
14,Savory,Wholesale,2,40999.92
8,Foods,DTS,2,36387.36
3,Biscuits,Wholesale,1,20645.11
1,Biscuits,C-Stores,1,11346.32
12,Savory,DTS,2,-2709.84
0,Biscuits,Away From Home,2,-2698.71
9,Foods,Organized Trade,1,-2373.39
2,Biscuits,DTS,1,-2244.97
10,Savory,Away From Home,2,-2227.87


In [4]:
# Simular columnas de tránsito logístico internacional/nacional para la Política 37
np.random.seed(10)
df_conciliacion['Fecha_Despacho'] = df_conciliacion['Fecha']

# Inyectar una fecha de entrega aleatoria (entre 1 y 15 días después del despacho)
dias_transito = [np.random.randint(1, 16) for _ in range(len(df_conciliacion))]
df_conciliacion['Fecha_Entrega'] = df_conciliacion['Fecha_Despacho'] + pd.to_timedelta(dias_transito, unit='D')

# Calcular el Lead Time real en días
df_conciliacion['Lead_Time_Dias'] = (df_conciliacion['Fecha_Entrega'] - df_conciliacion['Fecha_Despacho']).dt.days

# Definir la regla de la Política Global 37: Si el Lead Time supera los 10 días, requiere provisión/control
df_conciliacion['Alerta_Politica_37'] = np.where(df_conciliacion['Lead_Time_Dias'] > 10, 'Excede Lead Time Máximo', 'Dentro de Política')

print("🚚 REPORTE ESTADÍSTICO DE TRÁNSITO NACIONAL (POLÍTICA 37):")
print(df_conciliacion['Alerta_Politica_37'].value_counts())
print("\nMuestra de transacciones críticas en tránsito:")
df_conciliacion[df_conciliacion['Alerta_Politica_37'] == 'Excede Lead Time Máximo'][['ID_Transaccion', 'Unidad_Negocio', 'Lead_Time_Dias', 'Alerta_Politica_37']].head()


🚚 REPORTE ESTADÍSTICO DE TRÁNSITO NACIONAL (POLÍTICA 37):
Alerta_Politica_37
Dentro de Política         692
Excede Lead Time Máximo    308
Name: count, dtype: int64

Muestra de transacciones críticas en tránsito:


,ID_Transaccion,Unidad_Negocio,Lead_Time_Dias,Alerta_Politica_37
1,TX-100001,Savory,14,Excede Lead Time Máximo
5,TX-100005,Savory,12,Excede Lead Time Máximo
6,TX-100006,Foods,13,Excede Lead Time Máximo
8,TX-100008,Savory,14,Excede Lead Time Máximo
10,TX-100010,Biscuits,14,Excede Lead Time Máximo


In [5]:
# Guardar como CSV (más compatible y rápido)
df_conciliacion.to_csv('Reporte_Consolidado_Financiero.csv', index=False)
print("💾 Archivo generado en el sistema de Colab.")


💾 Archivo generado en el sistema de Colab.
